In [0]:
df_apolice   = spark.read.format("delta").table("bronze.apolice")
df_carro     = spark.read.format("delta").table("bronze.carro")
df_cliente   = spark.read.format("delta").table("bronze.cliente")
df_endereco  = spark.read.format("delta").table("bronze.endereco")
df_estado    = spark.read.format("delta").table("bronze.estado")
df_marca     = spark.read.format("delta").table("bronze.marca")
df_modelo    = spark.read.format("delta").table("bronze.modelo")
df_municipio = spark.read.format("delta").table("bronze.municipio")
df_regiao    = spark.read.format("delta").table("bronze.regiao")
df_sinistro  = spark.read.format("delta").table("bronze.sinistro")
df_telefone  = spark.read.format("delta").table("bronze.telefone")
 
print("✅ DataFrames lidos do Bronze!")


In [0]:
from pyspark.sql.functions import current_timestamp, lit
 
df_apolice   = df_apolice.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("apolice"))
df_carro     = df_carro.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("carro"))
df_cliente   = df_cliente.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("cliente"))
df_endereco  = df_endereco.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("endereco"))
df_estado    = df_estado.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("estado"))
df_marca     = df_marca.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("marca"))
df_modelo    = df_modelo.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("modelo"))
df_municipio = df_municipio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("municipio"))
df_regiao    = df_regiao.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("regiao"))
df_sinistro  = df_sinistro.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("sinistro"))
df_telefone  = df_telefone.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("telefone"))
 
print("✅ Metadados Silver adicionados!")


In [0]:
from pyspark.sql import functions as F
 
# ---------- Helpers ----------
def _apply_name_rules(colname: str) -> str:
    """Regras de renome: upper + expansão de prefixos/siglas."""
    n = colname.upper()
    n = n.replace("CD_",  "CODIGO_")
    n = n.replace("VL_",  "VALOR_")
    n = n.replace("DT_",  "DATA_")
    n = n.replace("NM_",  "NOME_")
    n = n.replace("DS_",  "DESCRICAO_")
    n = n.replace("NR_",  "NUMERO_")
    n = n.replace("_UF",  "_UNIDADE_FEDERATIVA")
    return n
 
def _safe_drop(df, cols):
    """Remove colunas somente se existirem no DataFrame."""
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df
 
def renomear_colunas_managed(src_fqn: str, dest_fqn: str = None):
    """
    Lê uma managed table (Delta) do metastore, aplica regras de Data Quality,
    ajusta colunas de auditoria e salva como managed table via saveAsTable.
    
    Args:
        src_fqn  : 'schema.tabela' de origem (ex.: 'bronze.apolice')
        dest_fqn : 'schema.tabela' de destino; se None, sobrescreve a origem
    """
    dest_fqn = dest_fqn or src_fqn
 
    # Lê como TABELA MANAGED
    df = spark.read.format("delta").table(src_fqn)
 
    # Renomeia todas as colunas de uma vez (evita conflito de rename em loop)
    new_cols = [_apply_name_rules(c) for c in df.columns]
    df = df.toDF(*new_cols)
 
    # Remove colunas de auditoria do Bronze (serão substituídas pelas do Silver)
    df = _safe_drop(df, ["DATA_HORA_BRONZE", "NOME_ARQUIVO"])
 
    # Adiciona colunas de auditoria Silver
    df = (df
          .withColumn("NOME_ARQUIVO_BRONZE", F.lit(src_fqn))         # rastreabilidade
          .withColumn("DATA_ARQUIVO_SILVER", F.current_timestamp())   # timestamp de processamento
         )
 
    # Salva como Managed Table no Silver (sobrescrevendo se existir)
    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(dest_fqn))
 
    print(f"  ✓ {src_fqn} → {dest_fqn}")
    return dest_fqn


In [0]:
print("Aplicando Data Quality e salvando no Silver...")
 
renomear_colunas_managed("bronze.apolice",   "silver.apolice")
renomear_colunas_managed("bronze.carro",     "silver.carro")
renomear_colunas_managed("bronze.cliente",   "silver.cliente")
renomear_colunas_managed("bronze.endereco",  "silver.endereco")
renomear_colunas_managed("bronze.estado",    "silver.estado")
renomear_colunas_managed("bronze.marca",     "silver.marca")
renomear_colunas_managed("bronze.modelo",    "silver.modelo")
renomear_colunas_managed("bronze.municipio", "silver.municipio")
renomear_colunas_managed("bronze.regiao",    "silver.regiao")
renomear_colunas_managed("bronze.sinistro",  "silver.sinistro")
renomear_colunas_managed("bronze.telefone",  "silver.telefone")
 
print("\n✅ Todas as tabelas salvas no schema Silver (Delta Lake) com Data Quality!")
 


In [0]:
%sql
SHOW TABLES IN silver

In [0]:
%sql
DESCRIBE DETAIL silver.apolice;
DESCRIBE EXTENDED silver.apolice;